In [52]:
48

48

In [53]:
import os
from dotenv import load_dotenv

In [54]:
#  langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings

In [55]:
text = """
    LangChain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.

"""


recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, chunk_overlap=10,
)

chunks = recursive_splitter.split_text(text)

print(f"Number of chunks: {len(chunks)}")
# print(f"Chunks: {chunks}")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk}")


Number of chunks: 5
Chunk 0: LangChain is a framework for developing
Chunk 1: applications powered by language models. It can
Chunk 2: It can be used for chatbots, Generative
Chunk 3: Question-Answering (GQA), summarization, and much
Chunk 4: and much more.


In [56]:
load_dotenv()

True

In [57]:
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

print("ENV Variables loaded successfully:")

ENV Variables loaded successfully:


In [58]:
dataFilePath = os.path.join("data", "hr_policy.txt") # reading from data folder

In [59]:
loader = TextLoader(dataFilePath, encoding="utf-8")
documents = loader.load()

In [60]:
type(documents)
len(documents)

1

In [61]:
documents[0].page_content[:50]  # print first 500 characters of the first document

'COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Ha'

In [62]:
documents[0].metadata  # print metadata of the first document

{'source': 'data/hr_policy.txt'}

In [83]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500 , chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(chunks)  # print first chunk

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [84]:
len(chunks)  # print number of chunks

9

In [85]:
print(chunks[0])  # print first chunk conten

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data/hr_policy.txt'}


In [86]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

In [87]:

embeddings_model.model_name

'jina-embeddings-v2-base-en'

In [88]:
from langchain_community.vectorstores import FAISS

In [89]:
vector_store = FAISS.from_documents(chunks, embeddings_model)

In [90]:
print ( f"Number of vectors in the vector store: {vector_store.index.ntotal}" )  # print number of vectors in the vector store

Number of vectors in the vector store: 9


In [91]:
test_query = "How many sick leaves are allowed in a year?"

In [92]:

top_matches = vector_store.similarity_search(test_query, k=2)  # retrieve top 2 similar documents from the vector store

In [93]:
print(top_matches)  # print the top matches

[Document(id='c4f94ee8-7991-43ec-8e4e-8de5dcf1c472', metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='ae7da7e3-924d-4325-ae79-c12cfb911700', metadata={'source': 'data/hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.')]


In [94]:
for i, match in enumerate(top_matches):
    print(f"Match {i}: {match.page_content}")  # print each match

Match 0: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Match 1: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


Retrieve Tool

In [95]:
vector_store 

In [97]:
retrieve_tool = vector_store.as_retriever(search_kwargs={"k": 3}) # create a retriever from the vector store that returns the top 3 most similar documents

def search_hr_policy(question :str)-> str:
    """

    Args:
        question (str): _description_

    Returns:
        str: _description_
    Search the HR policy document for the information about the leave , work from home,
    probation period , notice period , reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chunks = retrieve_tool.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

LLM

In [99]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.7
)

In [104]:
llm.model_name  

'openai/gpt-oss-120b'

In [110]:
groq_call = llm.invoke("What is the notice period for resignation?")
print(groq_call.content)

The length of the notice period you must give when you resign depends on a few key factors:

| Factor | Typical Notice Requirement |
|--------|----------------------------|
| **Employment contract or offer letter** | Whatever is written in the contract (e.g., 2 weeks, 30 days, 60 days). This is usually the controlling rule. |
| **Local labor laws** | Many countries set a statutory minimum (often “2 weeks” for non‑managerial staff). If the contract is silent, the legal minimum applies. |
| **Job level / seniority** | • Entry‑level / junior staff: 1–2 weeks<br>• Mid‑level / professional staff: 2–4 weeks<br>• Managerial / senior/executive: 30–90 days (or more) |
| **Industry norms** | • Retail / hospitality: 1 week to 2 weeks<br>• Tech / professional services: 2 weeks to 1 month<br>• Finance / law / academia: 30 days to 3 months |
| **Company policy / handbook** | Some firms have a standard “policy notice” that applies when the contract is silent. Check the employee handbook or intranet. 

In [116]:
groq_call.response_metadata.get("model_provider")   

'groq'

In [117]:
from langchain.agents import create_agent  

In [120]:
hr_assistant_agent = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
        You are a friendly and helpful HR assistant working for Yashree Corporation.
        Always use search_hr_policy tool to look up facts before answering.
        If the answer is not in the search results, say "I don't know" or "I am not sure"
        instead of guessing. 
        Dont entertain any other questions which are off topic and if you dont understand 
    """,
)

In [ ]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant_agent.invoke(
            {"messages": 
                [
                    {
                        "role": "   user",
                        "content": question
                    }
                ]
            }
            
    )
    


    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [124]:
response =  ask_hr_assistant("What is the policy  ?")

QUESTION: What is the policy  ?
------------------------------------------------------------
ANSWER: Could you let me know which specific HR policy you’d like information about (e.g., leave, work‑from‑home, probation period, notice period, reimbursement, code of conduct, holidays, or exit process)?

